# Project: Movie Recommender from Scratch (MovieLens)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/03_Machine_Learning/projects/other/movielens_recommender_project.ipynb)

End-to-end recommender on your local MovieLens data: popularity baseline -> item-item collaborative filtering -> matrix-factorization scoring, with honest train/test evaluation.

Builds directly on `02_Data_Analysis_EDA/movies_ratings_eda.ipynb`.

## 1. Load + split (time-aware!)

In [ ]:
import numpy as np, pandas as pd

movies = pd.read_csv("../../../content/movies.csv")
ratings = pd.read_csv("../../../content/ratings.csv")
print(movies.shape, ratings.shape)

ratings = ratings.sort_values("timestamp")
k = int(len(ratings) * 0.8)
train, test = ratings.iloc[:k], ratings.iloc[k:]
print(f"train {len(train):,} | test {len(test):,}")

# never recommend movies unseen in training
known_movies = set(train.movieId.unique())
test = test[test.movieId.isin(known_movies)]

Time-aware splitting mimics production: predict the FUTURE, never leak it.

## 2. Baseline 1 - popularity (Bayesian weighted)

In [ ]:
C, m = train.rating.mean(), 25
pop = train.groupby("movieId").rating.agg(["mean", "count"])
pop["score"] = ((pop["count"] * pop["mean"] + m * C) / (pop["count"] + m))
top_popular = pop.sort_values("score", ascending=False).head(10) \
                .merge(movies, on="movieId")
top_popular[["title", "mean", "count", "score"]]

## 3. Item-item collaborative filtering

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

user_movie = train.pivot_table(index="userId", columns="movieId",
                               values="rating").fillna(0)
item_sim = cosine_similarity(user_movie.T)
np.fill_diagonal(item_sim, 0)                    # a movie is not its own neighbor
sim_df = pd.DataFrame(item_sim, index=user_movie.columns, columns=user_movie.columns)

def similar(movie_id, n=6):
    s = sim_df[movie_id].nlargest(n)
    return movies.set_index("movieId").loc[s.index, "title"].assign(similarity=s.values.round(3))

title_id = movies.set_index("title").movieId
mid = int(title_id["Toy Story (1995)"])
print("Because you watched Toy Story (1995):")
similar(mid)

## 4. Matrix factorization scoring (TruncatedSVD)

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler

X = user_movie.values
means = X.mean(axis=1, keepdims=True)
Xc = X - means                                   # remove user bias

svd = TruncatedSVD(n_components=50, random_state=42)
latent = svd.fit_transform(Xc)
reconstructed = means + svd.inverse_transform(latent)
pred_df = pd.DataFrame(reconstructed, index=user_movie.index,
                       columns=user_movie.columns)
print(f"{svd.n_components} latent factors explain "
      f"{svd.explained_variance_ratio_.sum():.0%} of rating variance")

In [ ]:
def recommend(user_id, n=10):
    seen = set(train.loc[train.userId == user_id, "movieId"])
    preds = pred_df.loc[user_id].drop(index=[i for i in seen if i in pred_df.columns],
                                      errors="ignore")
    top = preds.nlargest(n).index
    return movies.set_index("movieId").loc[top, "title"]

print("Top-10 for user 42:")
recommend(42)

## 5. Evaluate: RMSE + Precision@10

In [ ]:
test_rated = test[test.userId.isin(user_movie.index)]
sample = test_rated.sample(4000, random_state=42)

def rmse_subset(df_t):
    errs = []
    for u, m_i, r in zip(df_t.userId, df_t.movieId, df_t.rating):
        if m_i in pred_df.columns:
            errs.append((pred_df.at[u, m_i] - r) ** 2)
    return float(np.sqrt(np.mean(errs)))

svd_rmse = rmse_subset(sample)
naive_rmse = np.sqrt(((test.rating - C) ** 2).mean())
print(f"predict-global-mean RMSE : {naive_rmse:.3f}")
print(f"SVD recommender RMSE     : {svd_rmse:.3f}")

hits = 0; users_eval = 0
for uid, grp in test.groupby("userId"):
    if uid not in pred_df.index or len(grp) < 5:
        continue
    good = set(grp.loc[grp.rating >= 4, "movieId"]) & set(user_movie.columns)
    if not good:
        continue
    rec_top = set(pred_df.loc[uid].drop(
        index=[i for i in set(train.loc[train.userId == uid, 'movieId'])
               if i in pred_df.columns], errors='ignore').nlargest(10).index)
    hits += len(rec_top & good) / 10
    users_eval += 1
print(f"Precision@10 (hit-rate vs >=4-star test items): {hits / max(users_eval,1):.3f}")

## Where to scale up
| Upgrade | Tool |
|---|---|
| bigger data, implicit feedback | `implicit` ALS library |
| deep match model | two-tower retrieval (TensorFlow Recommenders) |
| side features (genres, tags) | LightFM hybrid model |
| cold start | fall back to popularity/content similarity |

Key lesson from evaluation: always compare against the dumbest sensible baseline - complexity must earn its keep.